In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from os import listdir

from physics_definitions import load_physics_definitions, visible_limits

In [5]:
ROOT = Path.cwd().parent

In [8]:
RUN_DIR = ROOT / "outputs" / listdir(ROOT / "outputs")[0]
RESULT_DIR = ROOT / "results"
OUTPUT_DIR = ROOT / "plot_potentials" / "graphs_OOD"

In [9]:
suites = json.loads((RUN_DIR / "ood_suite_manifest.json").read_text(encoding="utf-8"))
family_summary = pd.read_csv(RESULT_DIR / "01_metrics" / "family_summary.csv")
metrics = pd.read_csv(RESULT_DIR / "01_metrics" / "per_state_metrics.csv")
targets = family_summary[family_summary["category"] != "ID_test"]
selected = []
for row in targets.itertuples(index=False):
    candidates = metrics[(metrics.category == row.category) & (metrics.family == row.family)]
    group_fidelity = candidates.groupby("group_id", as_index=False).fidelity.mean()
    nearest = group_fidelity.iloc[(group_fidelity.fidelity - row.fidelity).abs().argmin()]
    manifest_rows = suites[row.category]
    record = next(item for item in manifest_rows if item["group_id"] == nearest.group_id)
    selected.append((row.category, row.family, record, float(nearest.fidelity)))

In [10]:
def reconstruct(record: dict, PotentialSpec, sample_potential_safely):
    left, right = map(float, record["domain"])
    geometry = record["geometry"]
    family = record["family"]
    singular_left = geometry == "singular_interval" or family in {"eckart", "radial_coulomb"}
    singular_right = geometry == "singular_interval"
    spec = PotentialSpec(
        family, record["parameters"], geometry, record["boundary_condition"], left, right,
        record.get("continuum_threshold"), singular_left, singular_right, "ood", record["group_id"],
    )
    count = int(record["grid_nodes"])
    if record.get("grid_variant") == "nonuniform_resolution":
        t = np.linspace(0.0, 1.0, count)
        mapped = 0.5 * (1 + np.sinh(0.7 * (2 * t - 1)) / np.sinh(0.7))
        x = left + (right - left) * mapped
    else:
        # The OOD record is visualized on its actual audit resolution and domain.
        x = np.linspace(left, right, count)
    values, valid, singular = sample_potential_safely(spec, x)
    category = record["category"]
    if category == "constant_potential_shift":
        values = values + float(record.get("constant_shift_from_parent") or 0.0)
    elif category == "valid_spatial_reflection":
        values = values[::-1].copy()
        valid = valid[::-1].copy()
        singular = singular[::-1].copy()
    return x, values, valid, singular

In [13]:
PHYSICS_STAGE = ROOT / "stages" / "s01_physics.py"
SELECTED_NAMES = {
    "ANALYTICAL_FAMILIES",
    "FUNCTIONAL_FAMILIES",
    "TRAIN_FAMILIES",
    "REPARAMETERIZED_SEEN_FAMILIES",
    "UNSEEN_ANALYTICAL_FAMILIES",
    "HELD_OUT_FAMILIES",
    "PotentialSpec",
    "_coefficients",
    "make_spec",
    "potential_value",
    "geometry_grid",
    "sample_potential_safely",
}

In [14]:
definitions = load_physics_definitions(PHYSICS_STAGE, SELECTED_NAMES)
PotentialSpec = definitions["PotentialSpec"]
sample_potential_safely = definitions["sample_potential_safely"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = []

In [15]:
for index, (category, family, record, fidelity) in enumerate(selected, 1):
    x, values, valid, singular = reconstruct(record, PotentialSpec, sample_potential_safely)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(x[valid], values[valid], color="tab:purple", linewidth=1.8)
    ax.scatter(x[valid], values[valid], s=5, color="tab:purple", alpha=0.18)
    ax.axhline(0.0, color="black", linewidth=0.7, alpha=0.55)
    if singular.any():
        ax.scatter(x[singular], values[singular], marker="x", color="tab:red", label="singular endpoint")
        ax.legend()
    ax.set_ylim(*visible_limits(values[valid]))
    ax.set_xlabel("x")
    ax.set_ylabel("V(x)")
    ax.set_title(f"{family.replace('_', ' ').title()} — {category.replace('_', ' ')}")
    ax.text(0.01, 0.02, f"Representative mean fidelity: {fidelity:.6f}\n{record['parameters']}",
            transform=ax.transAxes, fontsize=8, va="bottom", wrap=True)
    ax.grid(alpha=0.2)
    fig.tight_layout()
    filename = f"{index:02d}_{category}__{family}.png"
    fig.savefig(OUTPUT_DIR / filename, dpi=150)
    plt.close(fig)
    manifest.append({
        "file": filename, "category": category, "family": family,
        "representative_group_id": record["group_id"],
        "representative_mean_fidelity": fidelity, "parameters": record["parameters"],
        "geometry": record["geometry"], "domain": record["domain"],
        "grid_nodes": record["grid_nodes"], "grid_variant": record.get("grid_variant"),
        "constant_shift_from_parent": record.get("constant_shift_from_parent"),
    })